#1. Basic Tasks

##1. Create a table, make 3 changes to it (insert, update, insert), and use DESCRIBE HISTORY to review the resulting versions. 

In [0]:
create or replace table cyntexa_dev.bronze.sales_raw as
select *, current_timestamp() as ingestion_ts
from read_files('/Volumes/dev/demo/raw/sales.csv');

select * from cyntexa_dev.bronze.sales_raw;

insert into cyntexa_dev.bronze.sales_raw
values(51, 6, 1051, 4, 2, 1.44, 235.99, '2023-12-04', null, '2026-08-26 11:00:57.599');

update cyntexa_dev.bronze.sales_raw
set ingestion_ts = current_timestamp()
where order_id = 51;

insert into cyntexa_dev.bronze.sales_raw
values(52, 7, 1052, 5, 3, 2.50, 450.00, '2023-12-05', null, current_timestamp());

--describe history cyntexa_dev.bronze.sales_raw;

##2. Use COPY INTO to incrementally load 2 batches of files into a bronze table, confirming COPY INTO doesn't reprocess the first batch. 

In [0]:
-- Batch 1: Load sales2.csv using COPY INTO with explicit type casts
copy into cyntexa_dev.bronze.sales_raw
from (
  select
    order_id,
    customer_id,
    transaction_id,
    product_id,
    quantity,
    discount_amount,
    total_amount,
    to_date(order_date, 'dd/MM/yy') as order_date,
    null as _rescued_data,
    current_timestamp() as ingestion_ts
  from '/Volumes/dev/demo/raw/sales/sales2.csv'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true');
select * from cyntexa_dev.bronze.sales_raw;


-- Batch 2: Load sales3.csv using COPY INTO with explicit type casts
copy into cyntexa_dev.bronze.sales_raw
from (
  select
    order_id,
    customer_id,
    transaction_id,
    product_id,
    quantity,
    discount_amount,
    total_amount,
    to_date(order_date, 'dd/MM/yy') as order_date,
    null as _rescued_data,
    current_timestamp() as ingestion_ts
  from '/Volumes/dev/demo/raw/sales/sales3.csv'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true');
select * from cyntexa_dev.bronze.sales_raw;


##3. Query an old version of the table with both VERSION AS OF and TIMESTAMP AS OF. 

In [0]:
 describe history cyntexa_dev.bronze.sales_raw;
 select * from cyntexa_dev.bronze.sales_raw version as of 70;
 select * from cyntexa_dev.bronze.sales_raw timestamp as of '2026-08-26T12:26:22.000+00:00';

##4. Evolve the table's schema two ways: append a new column using mergeSchema, then change an existing column's type using overwriteSchema; document the difference in what each requires

In [0]:
%python
from pyspark.sql.functions import lit
# Create a DataFrame with an additional column(mergeschema)
df_with_new_column = spark.table("cyntexa_dev.bronze.sales_raw")
df_with_new_column = df_with_new_column.withColumn("sales_channel", lit("online"))

df_with_new_column.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("cyntexa_dev.bronze.sales_raw")


spark.table("cyntexa_dev.bronze.sales_raw").printSchema()

# Read current data and cast the column to new type(overwriteSchema)
df_with_type_change = spark.table("cyntexa_dev.bronze.sales_raw")
df_with_type_change = df_with_type_change.withColumn("quantity", df_with_type_change["quantity"].cast("bigint"))


df_with_type_change.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("cyntexa_dev.bronze.sales_raw")

spark.table("cyntexa_dev.bronze.sales_raw").printSchema()



`mergeSchema:`
  - Purpose: Add new columns to existing schema
  - Mode: Works with append mode
  - Safety: Non-destructive, keeps all existing columns and types
  - Requirement: option('mergeSchema', 'true')
  - Use when: Adding new fields to your data model

`overwriteSchema:`
  - Purpose: Replace entire schema (change types, remove columns, restructure)
  - Mode: Requires overwrite mode
  - Safety: Destructive, can drop columns not in new DataFrame
  - Requirement: option('overwriteSchema', 'true') + mode('overwrite')
  - Use when: Making breaking changes to schema (type changes, column removal)

## 5. Set up an Autoloader stream ingesting from a folder, then drop 2 more files into the folder and confirm they're picked up automatically. 


In [0]:
CREATE OR REFRESH STREAMING TABLE cyntexa_dev.bronze.sales_autoloader
AS
SELECT *,
  _metadata.file_path as source_file,
  _metadata.file_modification_time as file_modified_time,
  current_timestamp() as ingestion_ts
FROM STREAM read_files(
  '/Volumes/cyntexa_dev/bronze/raw/sales/',
  format => 'csv',
  header => true
);

-- Query the streaming table to see ingested data
SELECT * FROM cyntexa_dev.bronze.sales_autoloader;


##6. Use RESTORE to roll a table back to a version before a bad schema change, and describe what happens to the versions that were created after the point you restored to. 

In [0]:

DESCRIBE HISTORY cyntexa_dev.bronze.sales_raw;
SELECT * FROM cyntexa_dev.bronze.sales_raw;

RESTORE TABLE cyntexa_dev.bronze.sales_raw TO VERSION AS OF 71;

DESCRIBE HISTORY cyntexa_dev.bronze.sales_raw;
-- Step 4: Check the schema is restored
DESCRIBE TABLE cyntexa_dev.bronze.sales_raw;



**After the RESTORE point:**

1. **Versions are NOT deleted**: All versions created after version 71 (versions 72+) still exist
   in the Delta log and can still be queried using time travel.

2. **New RESTORE version created**: The RESTORE operation itself creates a NEW version 
   (e.g., version 73) that points back to the data and schema from version 71.

3. **Current table state**: The current table now reflects version 71's data and schema,
   but this is actually version 73 (a new version that's a copy of version 71).

4. **Time travel still works**: You can still query the "bad" versions:
   - SELECT * FROM table VERSION AS OF 72 -- still works
   - This is useful for auditing or recovering specific data if needed


5. **History is preserved**: DESCRIBE HISTORY will show all operations including the RESTORE,
   maintaining a complete audit trail of all changes to the table.


##7. Compare all four ingestion patterns covered (batch CTAS, COPY INTO, Autoloader, Lakeflow Declarative Pipelines) on cost, latency, and operational complexity, and recommend which one Cyntexa should use for a file source that arrives unpredictably throughout the day. 



## Comparison of Ingestion Patterns

| Pattern | Cost | Latency | Operational Complexity | Best For |
|---------|------|---------|----------------------|----------|
| **Batch CTAS** | Lowest (pay only when run) | High (manual trigger or scheduled) | Simple (single SQL statement) | One-time loads, known schedules |
| **COPY INTO** |Low (idempotent, incremental) |Medium (scheduled runs) | Medium (requires scheduling + file tracking) | Periodic batch ingestion with predictable timing |
| **Auto Loader** |Medium (continuous streaming) | Low (near real-time, seconds) | Medium (streaming job management) | Unpredictable file arrivals, low-latency requirements |
| **Lakeflow Declarative Pipelines** | Medium-High (managed infrastructure) | Low (automatic triggers) | Lowest (fully managed, auto-scaling) | Production pipelines with dependencies, governance needs |

---

## Detailed Analysis

### 1. **Batch CTAS** (CREATE TABLE AS SELECT)
- **Cost**: Cheapest per run — compute runs only when triggered
- **Latency**: Hours to days — requires manual trigger or job scheduling
- **Complexity**: Minimal — single SQL statement, no state management
- **Limitation**: Rewrites entire table each time; no incremental processing

### 2. **COPY INTO**
- **Cost**: Low — idempotent (skips already-processed files), only charges for new data
- **Latency**: Minutes to hours — depends on schedule frequency (e.g., every 15 min, hourly)
- **Complexity**: Medium — need to schedule runs, track processed files, handle format options
- **Limitation**: Not truly real-time; each run scans directory for new files

### 3. **Auto Loader** (Structured Streaming)
- **Cost**: Moderate — streaming cluster runs continuously or uses triggered batches
- **Latency**: Seconds to minutes — picks up files automatically as they arrive
- **Complexity**: Medium — requires streaming table/job management, checkpointing, monitoring
- **Strength**: Event-driven ingestion; scales automatically; efficient incremental processing

### 4. **Lakeflow Declarative Pipelines** (DLT)
- **Cost**: Moderate to high — managed compute with auto-scaling overhead
- **Latency**: Low — continuous or triggered refresh; automatic dependency resolution
- **Complexity**: Lowest operational burden — fully managed, built-in monitoring, data quality checks
- **Strength**: Production-grade orchestration, lineage, expectations, pipeline observability

---

## Recommendation for Cyntexa

**Use Auto Loader (Streaming Table)** for a file source that arrives **unpredictably throughout the day**.

### Why Auto Loader?

1. **Event-Driven**: Automatically detects and processes new files within seconds—no need to guess scheduling intervals
2. **Cost-Efficient for Unpredictable Patterns**: 
   - COPY INTO wastes money running on a schedule when no files arrive
   - Auto Loader only processes when files land (trigger mode) or runs efficiently in continuous mode
3. **Scalability**: Handles file discovery efficiently even with thousands of files (file notification mode)
4. **Schema Evolution**: Built-in support for schema inference and evolution (rescue data column)
5. **Exactly-Once Semantics**: Checkpointing ensures no duplicate processing

### Implementation Approach

```sql
CREATE OR REFRESH STREAMING TABLE target_table
AS
SELECT *,
  _metadata.file_path as source_file,
  _metadata.file_modification_time as file_modified_time,
  current_timestamp() as ingestion_ts
FROM STREAM read_files(
  '/path/to/volume/',
  format => 'csv',
  header => true,
  cloudFiles.inferColumnTypes => true
);
```

### When to Consider Lakeflow Instead

Upgrade to **Lakeflow Declarative Pipelines** if Cyntexa needs:
- Multiple downstream transformations with dependency management
- Built-in data quality validation (expectations)
- End-to-end pipeline observability and lineage
- Production SLA requirements with managed infrastructure

### Cost Optimization Tip

For Auto Loader, use **triggered execution mode** if sub-minute latency isn't required:
```python
# Runs Auto Loader in micro-batch mode (e.g., every 5 minutes)
spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "csv") \
  .trigger(availableNow=True)  # or .trigger(processingTime="5 minutes")
```

This balances cost (compute only when checking for files) with low latency (much faster than hourly COPY INTO).


##8. Design a recovery runbook: if a bad file corrupts the silver table at 2am, walk through the exact commands (DESCRIBE HISTORY, RESTORE or time travel + overwrite) an on-call engineer would run.


# Silver Table Corruption Recovery Runbook

## Scenario
A bad file has corrupted the silver table at 2am. This runbook provides step-by-step recovery commands.

---

## Step 1: Assess the Damage

### 1.1 Check the current table state
```sql
SELECT COUNT(*) as total_rows FROM cyntexa_dev.silver.target_table;
SELECT * FROM cyntexa_dev.silver.target_table;
```

### 1.2 Review the table history
```sql
DESCRIBE HISTORY cyntexa_dev.silver.target_table;
```
**Look for:**
- The most recent operation (timestamp around 2am)
- The operation type (WRITE, MERGE, STREAMING UPDATE)
- The number of rows added/updated
- The version number of the corruption

### 1.3 Identify the last known good version
- Find the version **before** the 2am corruption
- Note the version number and timestamp

---

## Step 2: Verify the Last Good Version

### 2.1 Query using VERSION AS OF
```sql
-- Replace <version_number> with the last good version from DESCRIBE HISTORY
SELECT COUNT(*) as total_rows 
FROM cyntexa_dev.silver.target_table VERSION AS OF <version_number>;

-- Spot-check the data quality
SELECT * FROM cyntexa_dev.silver.target_table VERSION AS OF <version_number> LIMIT 100;
```
---

## Step 3: Recovery Options

### **RESTORE**

```sql
-- Restore to the last known good version
RESTORE TABLE cyntexa_dev.silver.target_table TO VERSION AS OF <version_number>;
```

**What happens:**
- Creates a NEW version that points back to the good data
- Does NOT delete the corrupted versions (audit trail preserved)
- Fast - no data rewrite required
- Current table immediately reflects the restored state

---

## Step 4: Verify Recovery

### 4.1 Confirm table state
```sql
SELECT COUNT(*) as total_rows FROM cyntexa_dev.silver.target_table;
SELECT * FROM cyntexa_dev.silver.target_table;
```

### 4.2 Check history to confirm restore operation
```sql
DESCRIBE HISTORY cyntexa_dev.silver.target_table;
```
- You should see a new RESTORE operation (or WRITE if using Option B)
- Note the new version number

### 4.3 Validate data quality
```sql
-- Run any business-specific validation queries
SELECT 
  COUNT(*) as total_rows,
  MIN(order_date) as earliest_date,
  MAX(order_date) as latest_date,
  SUM(total_amount) as total_revenue
FROM cyntexa_dev.silver.target_table;
```

---

## Step 5: Investigate Root Cause

### 5.1 Examine the corrupted version
```sql
-- Query the bad version to understand what went wrong
SELECT * FROM cyntexa_dev.silver.target_table VERSION AS OF <corrupted_version>
WHERE <condition_that_indicates_corruption>
;
```

### 5.2 Check the ingestion source
```sql
-- If corruption came from a file in a volume
LIST '/Volumes/cyntexa_dev/silver/raw/incoming/';

-- Examine the file timestamp around 2am
-- Move or quarantine the bad file
```

### 5.3 Review upstream pipeline logs
- Check job runs, streaming query logs, or Auto Loader checkpoint
- Identify which file or batch caused the issue

---

## Step 6: Prevent Recurrence

### 6.1 Add data quality checks
```sql
-- Example: Add expectations in DLT or validation in notebook
CREATE OR REPLACE TABLE cyntexa_dev.silver.target_table (
  order_id INT NOT NULL,
  customer_id INT NOT NULL,
  total_amount DECIMAL(10,2) CHECK (total_amount > 0),
  order_date DATE NOT NULL
);
```
### 6.2 Document the incident
- Record the corrupted version number
- Note the root cause (bad file format, schema mismatch, etc.)
- Update runbook with any lessons learned

---
## Important Notes

- **Time Travel Retention**: Delta tables retain history for 30 days by default
  - Check with: `DESCRIBE DETAIL cyntexa_dev.silver.target_table`
  
- **VACUUM Caution**: If someone ran VACUUM recently, old versions may be gone
  - VACUUM permanently deletes old data files
  - Cannot time travel beyond a VACUUM operation

- **Downstream Impact**: After restore, downstream tables may need refresh
  - Identify downstream consumers: `SHOW CREATE TABLE cyntexa_dev.gold.downstream_table`
  - Manually trigger downstream jobs if needed


##9. (Data Analyst) Using DESCRIBE HISTORY, produce a 'data freshness' report showing how frequently a given table is actually updated, to validate an SLA claim made to a business stakeholder. 


In [0]:
-- Step 1: Get the full history of the table
DESCRIBE HISTORY cyntexa_dev.bronze.sales_raw;

-- Step 2: Calculate time between updates (data freshness metrics)
WITH update_history AS (
  SELECT 
    version,
    timestamp,
    operation,
    operationMetrics,
    LAG(timestamp) OVER (ORDER BY version) as previous_update,
    TIMESTAMPDIFF(MINUTE, LAG(timestamp) OVER (ORDER BY version), timestamp) as minutes_since_last_update,
    TIMESTAMPDIFF(HOUR, LAG(timestamp) OVER (ORDER BY version), timestamp) as hours_since_last_update
  FROM (
    DESCRIBE HISTORY cyntexa_dev.bronze.sales_raw
  )
  WHERE operation IN ('WRITE', 'MERGE', 'STREAMING UPDATE', 'INSERT', 'UPDATE', 'DELETE', 'COPY')
)
SELECT 
  version,
  timestamp,
  operation,
  previous_update,
  minutes_since_last_update,
  hours_since_last_update,
  CASE 
    WHEN hours_since_last_update IS NULL THEN 'First Update'
    WHEN hours_since_last_update <= 1 THEN 'Within SLA (< 1 hour)'
    WHEN hours_since_last_update <= 24 THEN 'Within SLA (< 1 day)'
    ELSE 'SLA Breach'
  END as sla_status
FROM update_history
ORDER BY version DESC;

-- Step 3: Summary statistics for SLA validation
WITH update_history AS (
  SELECT 
    version,
    timestamp,
    operation,
    LAG(timestamp) OVER (ORDER BY version) as previous_update,
    TIMESTAMPDIFF(MINUTE, LAG(timestamp) OVER (ORDER BY version), timestamp) as minutes_since_last_update,
    TIMESTAMPDIFF(HOUR, LAG(timestamp) OVER (ORDER BY version), timestamp) as hours_since_last_update
  FROM (
    DESCRIBE HISTORY cyntexa_dev.bronze.sales_raw
  )
  WHERE operation IN ('WRITE', 'MERGE', 'STREAMING UPDATE', 'INSERT', 'UPDATE', 'DELETE', 'COPY')
)
SELECT 
  COUNT(*) as total_updates,
  MIN(timestamp) as first_update,
  MAX(timestamp) as last_update,
  TIMESTAMPDIFF(DAY, MIN(timestamp), MAX(timestamp)) as days_of_history,
  ROUND(AVG(hours_since_last_update), 2) as avg_hours_between_updates,
  ROUND(MIN(hours_since_last_update), 2) as min_hours_between_updates,
  ROUND(MAX(hours_since_last_update), 2) as max_hours_between_updates,
  ROUND(STDDEV(hours_since_last_update), 2) as stddev_hours_between_updates,
  ROUND(PERCENTILE(hours_since_last_update, 0.5), 2) as median_hours_between_updates,
  ROUND(PERCENTILE(hours_since_last_update, 0.95), 2) as p95_hours_between_updates,
  -- SLA compliance (assuming SLA is daily updates)
  ROUND(100.0 * SUM(CASE WHEN hours_since_last_update <= 24 THEN 1 ELSE 0 END) / COUNT(*), 2) as pct_within_24h_sla
FROM update_history
WHERE hours_since_last_update IS NOT NULL;

-- Step 4: Update frequency by day of week
WITH update_history AS (
  SELECT 
    timestamp,
    operation,
    DATE_FORMAT(timestamp, 'EEEE') as day_of_week,
    EXTRACT(DAYOFWEEK FROM timestamp) as day_number
  FROM (
    DESCRIBE HISTORY cyntexa_dev.bronze.sales_raw
  )
  WHERE operation IN ('WRITE', 'MERGE', 'STREAMING UPDATE', 'INSERT', 'UPDATE', 'DELETE', 'COPY')
)
SELECT 
  day_of_week,
  COUNT(*) as update_count,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) as pct_of_total_updates
FROM update_history
GROUP BY day_of_week, day_number
ORDER BY day_number;

-- Step 5: Recent freshness check (last 7 days)
WITH recent_updates AS (
  SELECT 
    DATE(timestamp) as update_date,
    COUNT(*) as updates_per_day,
    STRING_AGG(DISTINCT operation, ', ') as operations
  FROM (
    DESCRIBE HISTORY cyntexa_dev.bronze.sales_raw
  )
  WHERE timestamp >= CURRENT_TIMESTAMP() - INTERVAL 7 DAYS
    AND operation IN ('WRITE', 'MERGE', 'STREAMING UPDATE', 'INSERT', 'UPDATE', 'DELETE', 'COPY')
  GROUP BY DATE(timestamp)
)
SELECT 
  update_date,
  updates_per_day,
  operations,
  CASE 
    WHEN updates_per_day >= 1 THEN 'SLA Met'
    ELSE 'SLA Breach'
  END as daily_sla_status
FROM recent_updates
ORDER BY update_date DESC;